# Impact alpha: event time versus clock time

This notebook defines the first corrected-data alpha experiment. It compares continuous fixed-event bars with five-minute bars built from the same `timestamp_direction_v1` events. The primary economic horizon is approximately 15--30 minutes, but event-time targets remain explicit bar offsets (`n+h`) and always retain their realized elapsed-time distribution.

This is development-sandbox research. It does not create a final holdout result.

## Research contract

- Shared corrected events are the only input.
- Event bars: 2,000, 4,000 and 8,000 events, continuous across months.
- Five-minute bars: non-overlapping and constructed from identical events.
- Five-minute target offsets: 3, 4, 5 and 6 bars.
- Primary event targets match total future event count to 4,000-event `n+3`, `n+4`, `n+5` and `n+6`; duration calibration is a sensitivity analysis.
- Buy and sell impact functions are fitted separately.
- Current-bar information may predict later bars; future flow may never be a feature.
- Model selection is based on chronological validation and net performance under cost scenarios.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import polars as pl

REPO = Path('/Users/petermillington/Research/market-impact-research')
ALPHA = REPO / 'alpha-search'
SHARED = REPO / 'shared-data'
sys.path.insert(0, str(ALPHA / 'src'))

from alpha_search.bars import (
    add_bar_horizon_targets,
    add_trailing_impact_features,
    build_fixed_event_bars,
    build_time_bars,
    marginal_impact_proxy,
)
from alpha_search.impact_model import (
    add_asymmetric_impact_features,
    fit_asymmetric_log_impact,
)

In [ ]:
# A full build scans hundreds of millions of events. Keep this false while editing.
RUN_BUILD = False
SYMBOL = 'BTCUSDT'
EVENT_BAR_SIZES = (2_000, 4_000, 8_000)
REFERENCE_EVENT_SIZE = 4_000
REFERENCE_EVENT_HORIZONS = (3, 4, 5, 6)
TIME_BAR_SIZE = '5m'
TIME_HORIZONS = (3, 4, 5, 6)
TARGET_MINUTES = (15, 20, 25, 30)
TRAILING_WINDOW = '1d'
FEATURE_VERSION = 'impact_alpha_v1'
OUTPUT_ROOT = SHARED / 'features' / 'alpha' / f'symbol={SYMBOL}' / FEATURE_VERSION

## Select validated partitions

A month is included only when its validation report and reconstructed-event file both exist. This prevents partially downloaded directories from entering an analysis.

In [ ]:
reports = sorted((SHARED / 'metadata' / SYMBOL).glob('????-??.json'))
months = [path.stem for path in reports]
event_paths = []
for month in months:
    year, month_number = month.split('-')
    path = (SHARED / 'events' / 'timestamp_direction_v1' / f'symbol={SYMBOL}'
            / f'year={year}' / f'month={month_number}' / 'part-000.parquet')
    if path.exists():
        event_paths.append(path)

assert len(event_paths) == len(reports), 'Validated metadata/event coverage differs.'
print(f'{len(event_paths)} validated months: {months[0]} through {months[-1]}')

In [ ]:
events = pl.scan_parquet(event_paths)
events.collect_schema()

## Build common bar caches

The bars include contemporaneously observable impact and imbalance fields, strictly trailing one-day normalization, future returns, and realized target duration. The generated files are reproducible alpha features, not a new raw-data layer.

In [ ]:
def write_bar_cache(lazy_bars, name):
    path = OUTPUT_ROOT / name / 'part-000.parquet'
    path.parent.mkdir(parents=True, exist_ok=True)
    lazy_bars.sink_parquet(path, compression='zstd', compression_level=6)
    return path

if RUN_BUILD:
    time_bars = add_trailing_impact_features(
        add_bar_horizon_targets(
            build_time_bars(events, every=TIME_BAR_SIZE, assume_ordered=True),
            horizons=TIME_HORIZONS,
        ),
        window=TRAILING_WINDOW,
    )
    print(write_bar_cache(time_bars, f'time_{TIME_BAR_SIZE}'))

    # Broad event horizons permit training-period calibration for every size.
    for size in EVENT_BAR_SIZES:
        event_horizons = tuple(range(1, 13))
        event_bars = add_trailing_impact_features(
            add_bar_horizon_targets(
                build_fixed_event_bars(events, events_per_bar=size, assume_ordered=True),
                horizons=event_horizons,
            ),
            window=TRAILING_WINDOW,
        )
        print(write_bar_cache(event_bars, f'event_{size}'))

## Match event offsets and inspect clock duration

The primary comparison holds approximately constant the number of future events represented by the target: 4,000-event offsets 3--6 are the reference. For other bar sizes, use the nearest integer offset to the same total event count. Realized duration quantiles are reported by calendar fold. Training-period duration matching is retained only as a sensitivity check because market activity changes substantially through time.

In [ ]:
# Development split. Re-estimate these boundaries when later data are added.
TRAIN_END_MS = int(datetime(2020, 9, 1, tzinfo=timezone.utc).timestamp() * 1_000)

def matched_event_horizon(events_per_bar, reference_horizon):
    raw = reference_horizon * REFERENCE_EVENT_SIZE / events_per_bar
    return max(1, int(raw + 0.5))

PRIMARY_EVENT_HORIZONS = {
    size: tuple(matched_event_horizon(size, h) for h in REFERENCE_EVENT_HORIZONS)
    for size in EVENT_BAR_SIZES
}
PRIMARY_EVENT_HORIZONS

def duration_table(path, horizons=range(1, 13)):
    frame = pl.scan_parquet(path).filter(pl.col('end_time_ms') < TRAIN_END_MS)
    rows = []
    for h in horizons:
        col = f'future_elapsed_{h}_minutes'
        stats = frame.select(
            pl.lit(h).alias('horizon'),
            pl.col(col).quantile(0.10).alias('p10_minutes'),
            pl.col(col).median().alias('median_minutes'),
            pl.col(col).quantile(0.90).alias('p90_minutes'),
        ).collect()
        rows.append(stats)
    return pl.concat(rows)

def closest_offsets(durations, target_minutes=TARGET_MINUTES):
    return pl.DataFrame({'target_minutes': target_minutes}, schema={'target_minutes': pl.Float64}).join_asof(
        durations.sort('median_minutes'),
        left_on='target_minutes',
        right_on='median_minutes',
        strategy='nearest',
    )

# Duration-matching sensitivity after caches exist:
# d4000 = duration_table(OUTPUT_ROOT / 'event_4000' / 'part-000.parquet')
# display(d4000)
# display(closest_offsets(d4000))

## First impact-alpha models

Fit these models independently for each construction and event size:

1. Baseline: signed imbalance, absolute imbalance, activity, volatility and pre-trend.
2. Asymmetric impact: separate buy and sell intercepts/slopes for `log(1 + normalised_imbalance / scale)`.
3. Impact residual: observed contemporaneous signed impact minus the training-fitted expected impact.
4. Marginal-impact proxy: derivative of the fitted side-specific impact curve in bps per additional BTC.
5. Persistence extension: lagged signed imbalance and causal trailing averages only.

The initial question is whether unusually high or low fitted marginal impact, conditional on market state and side, predicts the `n+h` return. Report gross return, cost break-even, turnover, elapsed-time distribution and performance by month. Do not use future flow as a predictor.

In [ ]:
def fit_and_apply_impact_model(cache_path):
    bars = pl.scan_parquet(cache_path)
    training = bars.filter(pl.col('end_time_ms') < TRAIN_END_MS)
    fit = fit_asymmetric_log_impact(training)
    enriched = add_asymmetric_impact_features(bars, fit)
    return fit, enriched

def marginal_impact_summary(enriched, horizon):
    target = f'future_return_{horizon}_bps'
    return (
        enriched
        .filter(pl.col('end_time_ms') < TRAIN_END_MS)
        .with_columns(
            pl.col('marginal_impact_bps_per_btc')
            .qcut([0.2, 0.4, 0.6, 0.8], labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
            .alias('marginal_impact_quintile')
        )
        .group_by(
            pl.col('signed_imbalance').sign().alias('side'),
            'marginal_impact_quintile',
        )
        .agg(
            pl.len().alias('observations'),
            pl.col('marginal_impact_bps_per_btc').mean(),
            pl.col('impact_residual').mean(),
            pl.col(target).mean().alias('mean_future_return_bps'),
            pl.col(target).std().alias('future_return_std_bps'),
        )
        .sort('side', 'marginal_impact_quintile')
        .collect()
    )

# Example after caches exist:
# fit_4000, impact_4000 = fit_and_apply_impact_model(
#     OUTPUT_ROOT / 'event_4000' / 'part-000.parquet'
# )
# print(fit_4000)
# display(marginal_impact_summary(impact_4000, horizon=4))

## Current sandbox evidence

The first training/validation/later-sandbox run is summarized in `../reports/initial-impact-alpha-results.md`. The strongest repeating candidates are (1) high marginal-impact buy continuation on 8,000-event bars and (2) reversal after unusually large sell-side impact residuals, clearest on five-minute bars at 25--30 minutes. Both are explicitly asymmetric and remain hypotheses rather than trading claims.

In [ ]:
TABLE_ROOT = ALPHA / 'reports' / 'tables'
horizon_mapping = pl.read_csv(TABLE_ROOT / 'impact_alpha_v1_horizon_mapping.csv')
asymmetric_fits = pl.read_csv(TABLE_ROOT / 'impact_alpha_v1_asymmetric_fits.csv')
quintile_summary = pl.read_csv(TABLE_ROOT / 'impact_alpha_v1_quintile_summary.csv')
extreme_contrasts = pl.read_csv(TABLE_ROOT / 'impact_alpha_v1_extreme_contrasts.csv')

display(asymmetric_fits)
display(horizon_mapping.filter(pl.col('period') == 'validation'))
display(extreme_contrasts.filter(pl.col('period') == 'validation'))

## Evaluation rules

- Fit transformations, impact curves and event-offset calibration on training data only.
- Purge split boundaries by the largest target offset.
- Use expanding or rolling chronological validation.
- Use day-block or week-block uncertainty estimates because targets overlap.
- Compare event and time bars using the same calendar folds and cost assumptions.
- Keep the wave hypothesis out of model selection until this baseline family is frozen.